# EDA Olist - ventas, clientes y experiencia

Este notebook valida los outputs SQL creados previamente y explora las primeras historias que podrian convertirse en dashboard.

La intencion es mantener una lectura clara:

1. revisar que los datos procesados cargan correctamente;
2. validar KPIs principales;
3. explorar ventas en el tiempo;
4. analizar categorias;
5. revisar recurrencia de clientes;
6. conectar entrega y satisfaccion.


## tl;dr

Este resumen se basa en las celdas ejecutadas del notebook:

- El dataset procesado contiene 96,478 ordenes entregadas entre 2016-09-15 y 2018-08-29.
- El product revenue total es 13.22M y el ticket promedio es 137.04.
- La recurrencia es baja: 2,801 clientes recurrentes sobre 93,358 clientes unicos, equivalente a 3.00%.
- Las categorias con mayor revenue son `health_beauty`, `watches_gifts`, `bed_bath_table`, `sports_leisure` y `computers_accessories`.
- La entrega tardia parece afectar fuertemente la satisfaccion: ordenes con 8+ dias de atraso tienen review promedio 1.71.

Estos puntos son hallazgos preliminares. Las recomendaciones finales se escribiran despues de revisar visualizaciones y construir el dashboard.


## 1. Contexto y metodo

Trabajaremos sobre los CSV generados por SQL en `data/processed/`.

Esto es importante porque el notebook no deberia repetir toda la logica de joins. SQL ya dejo listas las tablas con el grano correcto:

- `fact_orders_delivered.csv`: una fila por orden entregada.
- `fact_order_items_delivered.csv`: una fila por item vendido en orden entregada.
- outputs agregados como `monthly_sales.csv`, `category_performance.csv` y `delivery_satisfaction.csv`.

En este notebook usamos Python para validar, explorar y visualizar.


In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"


## 2. Cargar datos procesados

Cada archivo viene de una consulta SQL previa. Cargarlos desde `data/processed/` permite que el notebook, el dashboard y las validaciones usen las mismas definiciones.


In [2]:
fact_orders = pd.read_csv(DATA_DIR / "fact_orders_delivered.csv", parse_dates=["purchase_date", "purchase_week", "purchase_month"])
fact_items = pd.read_csv(DATA_DIR / "fact_order_items_delivered.csv", parse_dates=["purchase_date", "purchase_week", "purchase_month"])
kpi_summary = pd.read_csv(DATA_DIR / "kpi_summary.csv", parse_dates=["first_purchase_date", "last_purchase_date"])
monthly_sales = pd.read_csv(DATA_DIR / "monthly_sales.csv", parse_dates=["purchase_month"])
category_performance = pd.read_csv(DATA_DIR / "category_performance.csv")
customer_frequency = pd.read_csv(DATA_DIR / "customer_frequency_summary.csv")
customer_lifetime = pd.read_csv(DATA_DIR / "customer_lifetime.csv", parse_dates=["first_purchase_date", "last_purchase_date"])
delivery_satisfaction = pd.read_csv(DATA_DIR / "delivery_satisfaction.csv")
customer_state_summary = pd.read_csv(DATA_DIR / "customer_state_summary.csv")
payment_summary = pd.read_csv(DATA_DIR / "payment_summary.csv")

loaded_tables = {
    "fact_orders": fact_orders,
    "fact_items": fact_items,
    "kpi_summary": kpi_summary,
    "monthly_sales": monthly_sales,
    "category_performance": category_performance,
    "customer_frequency": customer_frequency,
    "customer_lifetime": customer_lifetime,
    "delivery_satisfaction": delivery_satisfaction,
    "customer_state_summary": customer_state_summary,
    "payment_summary": payment_summary,
}

pd.DataFrame(
    [
        {"tabla": name, "filas": len(df), "columnas": len(df.columns)}
        for name, df in loaded_tables.items()
    ]
)


,tabla,filas,columnas
0,fact_orders,96478,38
1,fact_items,110197,36
2,kpi_summary,1,9
3,monthly_sales,23,7
4,category_performance,74,11
5,customer_frequency,4,6
6,customer_lifetime,93358,11
7,delivery_satisfaction,6,7
8,customer_state_summary,27,8
9,payment_summary,11,6


## 3. Validaciones rapidas

Antes de interpretar graficos, revisamos que las dos tablas base mantengan el grano esperado:

- ordenes: una fila por `order_id`;
- items: una fila por combinacion `order_id + order_item_id`.

Tambien verificamos que el revenue total sea consistente entre ambas tablas.


In [3]:
validation_summary = {
    "ordenes_filas": len(fact_orders),
    "ordenes_distintas": fact_orders["order_id"].nunique(),
    "items_filas": len(fact_items),
    "items_distintos": fact_items[["order_id", "order_item_id"]].drop_duplicates().shape[0],
    "revenue_ordenes": round(fact_orders["product_revenue"].sum(), 2),
    "revenue_items": round(fact_items["item_revenue"].sum(), 2),
}

validation_summary


{'ordenes_filas': 96478,
 'ordenes_distintas': 96478,
 'items_filas': 110197,
 'items_distintos': 110197,
 'revenue_ordenes': np.float64(13221498.11),
 'revenue_items': np.float64(13221498.11)}

## 4. KPIs principales

Esta tabla resume el estado general del negocio considerando solo ordenes entregadas.


In [4]:
kpi_summary.T.rename(columns={0: "valor"})


,valor
delivered_orders,96478
product_revenue,13221498.11
average_order_value,137.04
unique_customers,93358
average_review_score,4.16
late_delivery_rate_pct,8.11
average_delivery_days,12.5
first_purchase_date,2016-09-15 00:00:00
last_purchase_date,2018-08-29 00:00:00


## 5. Evolucion mensual de ventas

Este grafico responde si el negocio crece, cae o muestra estacionalidad. Para el dashboard, esta sera una visualizacion central del overview.


In [5]:
fig = px.line(
    monthly_sales,
    x="purchase_month",
    y="product_revenue",
    markers=True,
    title="Revenue mensual de ordenes entregadas",
    labels={"purchase_month": "Mes", "product_revenue": "Product revenue"},
)
fig.update_layout(yaxis_tickprefix="R$ ", hovermode="x unified")
fig.show()


## 6. Ordenes y ticket promedio

Revenue puede subir por mas ordenes, por mayor ticket promedio, o por ambos. Por eso revisamos volumen y AOV juntos.


In [6]:
fig = go.Figure()
fig.add_bar(
    x=monthly_sales["purchase_month"],
    y=monthly_sales["delivered_orders"],
    name="Ordenes entregadas",
    yaxis="y",
)
fig.add_trace(
    go.Scatter(
        x=monthly_sales["purchase_month"],
        y=monthly_sales["average_order_value"],
        name="AOV",
        mode="lines+markers",
        yaxis="y2",
    )
)
fig.update_layout(
    title="Ordenes entregadas y ticket promedio por mes",
    xaxis_title="Mes",
    yaxis={"title": "Ordenes"},
    yaxis2={"title": "AOV", "overlaying": "y", "side": "right", "tickprefix": "R$ "},
    legend={"orientation": "h"},
    hovermode="x unified",
)
fig.show()


## 7. Categorias con mayor revenue

Para priorizar acciones comerciales, no basta con saber el revenue total. Necesitamos entender que categorias lo explican y si el volumen se concentra en pocas de ellas.


In [7]:
top_categories = category_performance.head(12).copy()

fig = px.bar(
    top_categories.sort_values("product_revenue"),
    x="product_revenue",
    y="category",
    orientation="h",
    title="Top categorias por product revenue",
    labels={"product_revenue": "Product revenue", "category": "Categoria"},
    hover_data=["delivered_orders", "items_sold", "average_review_score"],
)
fig.update_layout(xaxis_tickprefix="R$ ")
fig.show()

top_categories[["category", "product_revenue", "delivered_orders", "items_sold", "average_item_revenue", "average_review_score"]]


,category,product_revenue,delivered_orders,items_sold,average_item_revenue,average_review_score
0,health_beauty,1233131.72,8647,9465,130.28,4.19
1,watches_gifts,1166176.98,5495,5859,199.04,4.07
2,bed_bath_table,1023434.76,9272,10953,93.44,3.92
3,sports_leisure,954852.55,7530,8431,113.25,4.17
4,computers_accessories,888724.61,6530,7644,116.26,3.99
5,furniture_decor,711927.69,6307,8160,87.25,3.95
6,housewares,615628.69,5743,6795,90.60,4.11
7,cool_stuff,610204.10,3559,3718,164.12,4.19
8,auto,578966.65,3810,4140,139.85,4.12
9,toys,471286.48,3804,4030,116.94,4.21


## 8. Concentracion de revenue por categoria

Un Pareto ayuda a ver si pocas categorias explican gran parte del revenue. Esta lectura es util para decidir foco comercial.


In [8]:
category_pareto = category_performance.sort_values("product_revenue", ascending=False).copy()
category_pareto["revenue_share"] = category_pareto["product_revenue"] / category_pareto["product_revenue"].sum()
category_pareto["cumulative_revenue_share"] = category_pareto["revenue_share"].cumsum()

fig = go.Figure()
fig.add_bar(
    x=category_pareto["category"].head(20),
    y=category_pareto["product_revenue"].head(20),
    name="Revenue",
)
fig.add_trace(
    go.Scatter(
        x=category_pareto["category"].head(20),
        y=category_pareto["cumulative_revenue_share"].head(20),
        name="Revenue acumulado",
        mode="lines+markers",
        yaxis="y2",
    )
)
fig.update_layout(
    title="Concentracion de revenue por categoria",
    xaxis_title="Categoria",
    yaxis={"title": "Product revenue", "tickprefix": "R$ "},
    yaxis2={"title": "Share acumulado", "overlaying": "y", "side": "right", "tickformat": ".0%"},
    legend={"orientation": "h"},
)
fig.show()


## 9. Recurrencia de clientes

La recurrencia indica si el negocio depende casi totalmente de nuevos compradores o si logra que una parte relevante vuelva a comprar.


In [9]:
fig = px.bar(
    customer_frequency,
    x="frequency_segment",
    y="customers",
    title="Clientes por frecuencia de compra",
    labels={"frequency_segment": "Frecuencia", "customers": "Clientes"},
    text="customer_share_pct",
    hover_data=["product_revenue", "revenue_share_pct", "average_order_value"],
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.show()

customer_frequency


,frequency_segment,customers,customer_share_pct,product_revenue,revenue_share_pct,average_order_value
0,1 order,90557,97.00,12493089.36,94.49,137.96
1,2 orders,2573,2.76,631274.90,4.77,122.67
2,3 orders,181,0.19,65974.42,0.50,121.50
3,4+ orders,47,0.05,31159.43,0.24,144.21


## 10. Valor de clientes

Este corte es exploratorio. Sirve para mirar si existen clientes de alto valor y si vale la pena crear una segmentacion mas fina en la siguiente version.


In [10]:
value_summary = (
    customer_lifetime
    .groupby("value_segment", as_index=False)
    .agg(
        customers=("customer_unique_id", "count"),
        product_revenue=("customer_revenue", "sum"),
        average_customer_revenue=("customer_revenue", "mean"),
        repeat_rate=("is_repeat_customer", "mean"),
    )
)
value_summary["product_revenue"] = value_summary["product_revenue"].round(2)
value_summary["average_customer_revenue"] = value_summary["average_customer_revenue"].round(2)
value_summary["repeat_rate_pct"] = (100 * value_summary["repeat_rate"]).round(2)
value_summary.drop(columns=["repeat_rate"])


,value_segment,customers,product_revenue,average_customer_revenue,repeat_rate_pct
0,high_value,3634,3375120.72,928.76,7.73
1,low_value,69418,4863264.60,70.06,1.57
2,mid_value,20306,4983112.79,245.40,7.04


## 11. Entrega y satisfaccion

Esta es una de las historias mas fuertes del proyecto: conectar un indicador operacional con una medida de experiencia del cliente.


In [11]:
delivery_order = [
    "7+ days early",
    "1-6 days early",
    "on estimate",
    "1-3 days late",
    "4-7 days late",
    "8+ days late",
]

delivery_satisfaction["delivery_delay_segment"] = pd.Categorical(
    delivery_satisfaction["delivery_delay_segment"],
    categories=delivery_order,
    ordered=True,
)
delivery_satisfaction = delivery_satisfaction.sort_values("delivery_delay_segment")

fig = px.bar(
    delivery_satisfaction,
    x="delivery_delay_segment",
    y="average_review_score",
    title="Review score promedio segun atraso o adelanto de entrega",
    labels={"delivery_delay_segment": "Segmento de entrega", "average_review_score": "Review score promedio"},
    text="average_review_score",
    hover_data=["delivered_orders", "average_delivery_days", "average_delay_days"],
)
fig.update_yaxes(range=[0, 5])
fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

delivery_satisfaction


,delivery_delay_segment,delivered_orders,product_revenue,average_order_value,average_delivery_days,average_delay_days,average_review_score
0,7+ days early,76140,10446086.92,137.20,10.27,-15.30,4.31
1,1-6 days early,12504,1615241.50,129.18,14.19,-4.02,4.18
2,on estimate,1292,172996.17,133.90,19.17,0.00,4.04
3,1-3 days late,1870,250856.63,134.15,23.28,1.83,3.29
4,4-7 days late,1802,287685.63,159.65,28.18,5.52,2.11
5,8+ days late,2870,448631.26,156.32,44.46,19.58,1.71


## 12. Estados con mayor revenue y riesgo operacional

Esta tabla ayuda a priorizar regiones: no solo donde hay mas revenue, sino tambien donde la entrega o satisfaccion puede afectar la experiencia.


In [12]:
customer_state_summary.head(12)[
    [
        "customer_state",
        "product_revenue",
        "delivered_orders",
        "unique_customers",
        "average_order_value",
        "average_review_score",
        "late_delivery_rate_pct",
        "average_delivery_days",
    ]
]


,customer_state,product_revenue,delivered_orders,unique_customers,average_order_value,average_review_score,late_delivery_rate_pct,average_delivery_days
0,SP,5067633.16,40501,39156,125.12,4.25,5.89,8.70
1,RJ,1759651.13,12350,11917,142.48,3.97,13.47,15.24
2,MG,1552481.83,11354,11001,136.73,4.19,5.61,11.94
3,RS,728897.47,5345,5168,136.37,4.19,7.15,15.25
4,PR,666063.51,4923,4769,135.30,4.24,5.00,11.94
5,SC,507012.13,3546,3449,142.98,4.13,9.76,14.90
6,BA,493584.14,3256,3158,151.59,3.93,14.04,19.28
7,DF,296498.41,2080,2019,142.55,4.13,7.07,12.90
8,GO,282836.70,1957,1895,144.53,4.10,8.18,15.54
9,ES,268643.45,1995,1928,134.66,4.08,12.23,15.72


## 13. Pagos y ticket promedio

Este corte no sera necesariamente protagonista del dashboard, pero puede explicar diferencias de ticket y comportamiento de compra.


In [13]:
payment_summary.head(12)


,main_payment_type,installment_segment,delivered_orders,product_revenue,average_order_value,unique_customers
0,credit_card,7+ installments,11751,3539911.73,301.24,11471
1,credit_card,2-3 installments,22020,2502391.42,113.64,21667
2,credit_card,4-6 installments,15696,2463035.35,156.92,15423
3,boleto,1 installment,19191,2329807.64,121.40,18717
4,credit_card,1 installment,23358,1926118.19,82.46,22941
5,voucher,1 installment,2783,244916.95,88.00,2716
6,debit_card,1 installment,1484,177868.46,119.86,1469
7,voucher,2-3 installments,139,24777.53,178.26,138
8,voucher,4-6 installments,43,8659.97,201.39,43
9,voucher,7+ installments,12,3875.90,322.99,12


## 14. Takeaways preliminares

1. El negocio tiene suficiente historia temporal para mostrar tendencia mensual y semanal.
2. Las categorias top concentran una parte relevante del revenue, por lo que un analisis de mix comercial tiene sentido.
3. La recurrencia es baja y debe ser una pregunta central del caso.
4. La entrega tardia se asocia con una caida clara de review score, especialmente desde 4 dias de atraso.
5. El dashboard deberia priorizar cuatro vistas: overview, categorias, clientes y entrega/satisfaccion.

Proximo paso: convertir estas exploraciones en un dashboard Streamlit con filtros y graficos interactivos.
